# FIT5196 Assessment 1 — EDA Notebook

**Group:** Group001
**Members:** Echo Zhao · Jasmine · Yandu Wang · Shawn

This notebook reads the six standardised CSVs produced by `Group001_solution.ipynb` and
nothing else. It does not re-run any cleaning: every value here is a value that was exported,
validated and shipped. It is the reproducibility evidence for `Group001_EDA.pdf`.

## 0. Configuration and data loading

Load the six standardised CSVs produced by the solution notebook. Keep paths
relative/configurable and use explicit read options where the literal string
`NaN` must remain visible.


Two cells. The first mounts Google Drive and does nothing anywhere else, so the configuration
cell below stays free of environment-specific code. The second finds the six CSVs from a short
list of candidate folders — the first entry is the layout produced by unzipping the submission.

**This notebook needs only the six CSVs.** It never re-runs the cleaning pipeline, so nobody has
to run the solution notebook to work here.

In [ ]:
# Colab only. Does nothing elsewhere.
try:
    from google.colab import drive
    from pathlib import Path as _Path
    import os as _os

    drive.mount('/content/drive')
    _root = _Path('/content/drive/MyDrive')
    _found = sorted({p.parent for p in _root.rglob('Group001_orders_standardised.csv')})
    if len(_found) == 1:
        _os.chdir(_found[0])
        print('cwd:', _found[0])
    elif len(_found) > 1:
        # Never guess. Several copies of the six CSVs means someone re-ran the pipeline into
        # their own folder, and picking the first would silently analyse the wrong set.
        print('Several folders on this drive hold the six CSVs:')
        for _p in _found:
            print('   ', _p)
        raise SystemExit('Pick one: set OUTPUT_DIR by hand in the next cell, or leave only '
                         'the shared copy on the drive.')
    else:
        print('Six CSVs not found under MyDrive. Put them in a folder there, or upload them.')
except ImportError:
    pass   # not on Colab

In [ ]:
# --- Section 0: configuration ---
from pathlib import Path

GROUP_ID = 'Group001'
CSV_NAME = f'{GROUP_ID}_orders_standardised.csv'

# The six CSVs, wherever they are. The first candidate is the layout produced by
# unzipping the submission; the rest cover the shared-drive folders.
OUTPUT_CANDIDATES = [Path('outputs'), Path('.'), Path('../outputs'),
                     Path('02_Outputs'), Path('../02_Outputs'),
                     Path('../00_Master/outputs')]
OUTPUT_DIR = next((p for p in OUTPUT_CANDIDATES if (p / CSV_NAME).exists()), None)
if OUTPUT_DIR is None:                       # last resort: search downwards from here
    hit = next(Path('.').rglob(CSV_NAME), None)
    OUTPUT_DIR = hit.parent if hit else None
assert OUTPUT_DIR is not None, (
    f'{CSV_NAME} not found. Put the six CSVs in an "outputs" folder beside this notebook. '
    f'Tried: {[str(p) for p in OUTPUT_CANDIDATES]}')
print('reading from', OUTPUT_DIR.resolve().name)

# Which copy of the six CSVs is this? Four people can each produce a set; a short fingerprint
# makes it visible at a glance that every figure in this notebook was built on the same one.
import hashlib
print()
for _p in sorted(OUTPUT_DIR.glob(f'{GROUP_ID}_*_standardised.csv')):
    _h = hashlib.sha256(_p.read_bytes()).hexdigest()[:12]
    print(f'   {_h}  {_p.name}')

In [ ]:
# --- Section 0: load the six tables ---
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7.5, 4.2),
                     'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

TABLES = ['orders', 'order_items', 'customers', 'deliveries', 'products', 'product_reviews']

# keep_default_na=False so the literal three-character 'NaN' sentinel stays visible as text
# rather than becoming a missing value, which would silently change every denominator.
T = {t: pd.read_csv(OUTPUT_DIR / f'{GROUP_ID}_{t}_standardised.csv', keep_default_na=False)
     for t in TABLES}

orders          = T['orders']
order_items     = T['order_items']
customers       = T['customers']
deliveries      = T['deliveries']
products        = T['products']
product_reviews = T['product_reviews']

for t in TABLES:
    print(f'{t:16s} {len(T[t]):>7,} rows x {T[t].shape[1]:>2} cols')

### 0.1 The grain of each table, and the join guard

Every metric in this notebook is calculated at its intended grain. The tables are one-to-many
in three places, so a join made in the wrong order multiplies rows and inflates any sum taken
afterwards:

| Table | One row per | Rows |
|---|---|---|
| `orders` | order | 5,000 |
| `order_items` | item within an order (1–5 per order) | 15,685 |
| `customers` | customer | 500 |
| `deliveries` | order | 5,000 |
| `products` | product | 1,000 |
| `product_reviews` | reviewed order item | 7,000 |

`at_grain()` below is used on every join in this notebook. It states the row count expected
after the join and fails if the join changed it, so an inflated revenue figure cannot survive
to a chart.

In [ ]:
# --- Section 0.1: the guard every join in this notebook passes through ---

def at_grain(df, expected_rows, label):
    """Assert a join did not multiply rows, and say so in the output.

    A one-to-many join is not wrong in itself; summing a parent column after one is.
    Naming the expected row count at the join is what makes the difference visible.
    """
    assert len(df) == expected_rows, (
        f'{label}: {len(df):,} rows after the join, expected {expected_rows:,} — '
        f'the join multiplied rows, so any sum taken now is inflated')
    print(f'{label}: {len(df):,} rows, grain preserved')
    return df


# Numeric columns arrive as text because of keep_default_na=False. Cast where needed.
def num(series):
    return pd.to_numeric(series, errors='coerce')


for t in TABLES:
    key = T[t].columns[0]
    print(f'{t:16s} one row per {key:<16s} {T[t][key].is_unique}')

## 1. Context and data-preparation assurance

Briefly define the business context and data scope. Summarise 3-5 material
transformation decisions and 4-6 material validation results by citing stable
`MAP-...` and `VAL-...` IDs. Do not repeat the complete mapping or notebook.


*Owner: Yandu. Three to five transformation decisions and four to six validation results,
citing `MAP-` and `VAL-` IDs from the mapping CSV and the validation register. Do not repeat
either artefact — this is the short assurance narrative the report's section 2 is built from.*

Candidates, each already evidenced in the solution notebook:

- **The canonical-row rule** — normalise, then keep one row per business key (`VAL-FLOW-07`,
  `VAL-FLOW-09`, `VAL-FLOW-10`). 0 field disagreements, within source or across sources.
- **The monetary chain** — the five money fields are rebuilt rather than copied, and the
  discount is percentage points applied before delivery with GST divided out, not added
  (`VAL-ARITH-01` to `VAL-ARITH-04`; the control shows the wrong formula matches 0 of 5,000).
- **`delivery_note_clean` is a direct copy** — a structured category, not narrative
  (`VAL-TEXT-01b`, `MAP-deliveries-20`).
- **Extraction runs on the raw text, measurement on the cleaned text** (`VAL-TEXT-06`,
  `VAL-TEXT-14`, `VAL-TEXT-15`).
- **The sentinel is three characters** — `coupon_code` and `promo_code` carry it on the same
  3,127 orders and nothing else does (`VAL-TEXT-11`).

## 2. Assessed EDA visualisations

Submit 6-8 clearly labelled assessed figures. Across the set, cover all six
published categories, at least four tables and at least two valid relational
analyses. For each figure state the question, observation unit, denominator,
tables/join keys, interpretation and material limitation.


### Figure 1: Univariate distribution or composition

**Owner:** Yandu

**Question:** How is order value distributed across the canonical orders, and is it skewed enough that a mean would mislead?

**Observation unit and denominator:** One order. Denominator: all 5,000 canonical orders.

**Tables and join keys:** `orders` only — no join.

**How to build it:** A histogram or ECDF of `order_total`, with the median and mean both marked so the skew is visible rather than asserted.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 1 (Yandu) ---

### Figure 2: Bivariate relationship or group comparison

**Owner:** Echo

**Question:** Which product categories earn the most revenue, and does the ranking change between revenue and units sold?

**Observation unit and denominator:** One order line. Denominator: all 15,685 canonical order items.

**Tables and join keys:** `order_items` joined to `products` on `product_id`. Revenue is summed at line grain.

**How to build it:** **This figure carries the double-counting demonstration.** Summing `order_total` after joining up to `orders` multiplies every order by its cart size. Compute both, print both, and say which is right and why. Use `at_grain(..., len(order_items), ...)`.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 2 (Echo) ---

### Figure 3: Temporal pattern

**Owner:** Jasmine

**Question:** How do order volume and revenue move through the year, and is any month an outlier?

**Observation unit and denominator:** One order. Denominator: all 5,000 canonical orders, grouped by month.

**Tables and join keys:** `orders` only — no join.

**How to build it:** `order_timestamp` is 2018 throughout; `delivered_date` runs into January 2019 for orders placed late in December, so a temporal range constraint belongs on the order timestamp and would report 76 false failures on the delivery date.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 3 (Jasmine) ---

### Figure 4: Multivariate or segmented relationship (also: operational)

**Owner:** Jasmine

**Question:** Does the on-time rate differ by carrier and by service level, and do the two interact?

**Observation unit and denominator:** One delivery. Denominator: all 5,000 deliveries.

**Tables and join keys:** `deliveries` joined to `orders` on `order_id` — strictly one-to-one.

**How to build it:** Use `delay_days` or `on_time_in_full`, **not** `delivery_note_clean`: the note is the same partition as the outcome columns and adds nothing. A grouped bar or small-multiple panel; keep the denominator per carrier visible so a small carrier is not read as a trend.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 4 (Jasmine) ---

### Figure 5: Review or text behaviour

**Owner:** Shawn

**Question:** Do rating, review length and script relate to one another — do longer reviews rate lower, and do non-Latin reviews differ?

**Observation unit and denominator:** One review. Denominator: all 7,000 canonical reviews.

**Tables and join keys:** `product_reviews` only — no join.

**How to build it:** `review_length_chars`, `rating`, `contains_non_latin_script`. 271 reviews are non-Latin, so plot rates rather than counts or the group disappears. `verified_purchase` is `True` on every row and cannot carry a visualisation.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 5 (Shawn) ---

### Figure 6: Delivery or operational performance

**Owner:** Echo

**Question:** Do late deliveries attract lower ratings, and how large is the difference?

**Observation unit and denominator:** One review. Denominator: the 7,000 reviews, **not** the 5,000 orders.

**Tables and join keys:** `product_reviews` joined to `deliveries` on `order_id`.

**How to build it:** **The grain trap.** Reviews sit at order-item grain, so an order with three reviewed items contributes three rows. Print the row count before and after the join in the same cell and pass it through `at_grain`. 3,993 distinct orders carry a review, so an order-level denominator would be wrong twice over.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 6 (Echo) ---

### Figure 7: Segmented relationship

**Owner:** Yandu

**Question:** Which customer segments spend the most, and is the difference explained by order count or by order size?

**Observation unit and denominator:** One customer. Denominator: all 500 customers.

**Tables and join keys:** `customers` joined to an aggregate of `orders` by `customer_id`. Aggregate first, then join — the other order multiplies customers by their order count.

**How to build it:** Splitting total spend into orders-per-customer and value-per-order is what makes this a segmented relationship rather than a second revenue chart.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 7 (Yandu) ---

### Figure 8: Multivariate relationship

**Owner:** Shawn

**Question:** Does unit price relate to rating and to helpful votes, or are highly rated products simply the cheap ones?

**Observation unit and denominator:** One reviewed order item. Denominator: the 7,000 reviews.

**Tables and join keys:** `product_reviews` joined to `order_items` on `order_item_id`, then to `products` on `product_id`. Two joins, both one-to-one from the review side.

**How to build it:** Bin `unit_price` rather than scattering 7,000 points, and show the count per bin so a sparse expensive bin is not over-read.

**Interpretation and limitation:** *Replace after producing the figure — one sentence on what
the figure shows, one on a material limitation.*

In [ ]:
# --- Figure 8 (Shawn) ---

#### Coverage check

The specification requires the assessed set to cover all six published categories, use evidence
from at least four of the six output tables, and include at least two correct relational
analyses. This cell states which figure covers what, so the requirement is checked rather than
assumed.

In [ ]:
# --- Coverage of the assessed set, checked rather than assumed ---
COVERAGE = {
    1: dict(categories={'univariate'},                 tables={'orders'},                            joins=0),
    2: dict(categories={'bivariate'},                  tables={'order_items', 'products'},           joins=1),
    3: dict(categories={'temporal'},                   tables={'orders'},                            joins=0),
    4: dict(categories={'multivariate', 'operational'},tables={'deliveries', 'orders'},              joins=1),
    5: dict(categories={'text'},                       tables={'product_reviews'},                   joins=0),
    6: dict(categories={'operational', 'bivariate'},   tables={'product_reviews', 'deliveries'},     joins=1),
    7: dict(categories={'multivariate'},               tables={'customers', 'orders'},               joins=1),
    8: dict(categories={'multivariate'},               tables={'product_reviews', 'order_items', 'products'}, joins=2),
}

REQUIRED = {'univariate', 'bivariate', 'multivariate', 'temporal', 'text', 'operational'}
seen_categories = set().union(*(f['categories'] for f in COVERAGE.values()))
seen_tables     = set().union(*(f['tables'] for f in COVERAGE.values()))
join_count      = sum(1 for f in COVERAGE.values() if f['joins'] > 0)

print(f'figures            {len(COVERAGE)}   (6 to 8 required)')
print(f'categories covered {len(seen_categories)}/6  missing: {sorted(REQUIRED - seen_categories) or "none"}')
print(f'tables used        {len(seen_tables)}/6  ({", ".join(sorted(seen_tables))})')
print(f'relational figures {join_count}     (at least 2 required)')

assert 6 <= len(COVERAGE) <= 8
assert REQUIRED <= seen_categories
assert len(seen_tables) >= 4
assert join_count >= 2
print('\ncoverage requirements met')

## 3. Ten evidence-based findings

Write exactly ten numbered findings. Keep each concise and decision-focused.
Each must identify the evidence, grain, magnitude/denominator, business meaning,
an alternative explanation or uncertainty, and a proportionate implication.
One figure may support more than one genuinely distinct finding.


*Each finding cites an assessed figure or a reported statistic, and carries all five parts:
what was observed and at what grain · magnitude, denominator or sample size · why it matters ·
an alternative explanation or uncertainty · a proportionate implication.*

1. **Finding 1** *(Echo, from Figure 2)* — Replace.
2. **Finding 2** *(Echo, from Figure 6)* — Replace.
3. **Finding 3** *(Jasmine, from Figure 3)* — Replace.
4. **Finding 4** *(Jasmine, from Figure 4)* — Replace.
5. **Finding 5** *(Yandu, from Figure 1)* — Replace.
6. **Finding 6** *(Yandu, from Figure 7)* — Replace.
7. **Finding 7** *(Shawn, from Figure 5)* — Replace.
8. **Finding 8** *(Shawn, from Figure 8)* — Replace.
9. **Finding 9** *(Yandu, from the validation register — no figure needed)* — The two sources
   agree completely where they overlap: across the keys carried by both files and every shared
   column, there are 0 field disagreements after normalisation, and 0 differences between the
   two copies of any within-source duplicate (`VAL-FLOW-09`, `VAL-FLOW-10`). *Add magnitude,
   the alternative explanation — that agreement may reflect a common upstream system rather
   than independent capture — and the implication for trusting either source alone.*
10. **Finding 10** *(Yandu, from the validation register — no figure needed)* —
    `deliveries.delivery_note_clean` is the same partition, row for row with zero disagreements
    across all 5,000 deliveries, as `delay_reason == 'none'`, `on_time_in_full`,
    `delay_days == 0` and `delivered_date <= promised_date`. It carries no information the
    outcome columns do not already carry. *Implication: it is a redundant feature that would
    leak the target into ML question 1.*

## 4. Five future machine-learning questions

Write exactly five numbered questions covering at least two problem types.
Model training is not required. Keep each response compact enough for the
ten-page report.


### MLQ-1 (Jasmine): Will this order be delivered later than promised?

| Element | Response |
|---|---|
| EDA evidence | *Cite the figure or finding and its quantified observation* |
| Business decision | *Replace* |
| Problem type and analysis unit | classification · one order at the moment it is placed |
| Target or unsupervised objective | `on_time_in_full`, or `delivered_date > promised_date` |
| Decision-time predictors | Order timestamp, sales channel, service level, carrier, promised days, shipping distance, delivery cost, cart size and value, customer history |
| Validation split | Temporal split — train on earlier months, test on later, because a random split lets the model see the future of the same period |
| Evaluation metric | Recall at a fixed alert budget, since the decision is which orders to intervene on |
| Leakage, fairness or deployment risk | **Target leakage is the whole problem here.** `delay_days`, `on_time_in_full`, `delay_reason` and `delivery_note_clean` are all recorded after delivery and all encode the answer — Finding 10 shows the note alone reproduces the outcome exactly. `delivered_date` is equally unavailable at decision time. |

### MLQ-2 (Echo): What will this basket be worth when the customer checks out?

| Element | Response |
|---|---|
| EDA evidence | *Cite the figure or finding and its quantified observation* |
| Business decision | *Replace* |
| Problem type and analysis unit | regression · one order at the point the cart is opened |
| Target or unsupervised objective | `order_total` |
| Decision-time predictors | Customer history and segment, sales channel, time of day and month, items and categories currently in the cart |
| Validation split | Temporal split, with customers held out as a second check so the model is not simply memorising individual buyers |
| Evaluation metric | Mean absolute error in dollars, which is directly interpretable as the planning error |
| Leakage, fairness or deployment risk | `coupon_discount`, `tax_amount` and `order_price` are components of the target and are only known once the order is complete. Customer-level aggregates must be computed from orders strictly before the one being predicted. |

### MLQ-3 (Shawn): Which delivered items will attract a rating of 2 or below?

| Element | Response |
|---|---|
| EDA evidence | *Cite the figure or finding and its quantified observation* |
| Business decision | *Replace* |
| Problem type and analysis unit | classification · one delivered order item |
| Target or unsupervised objective | `rating <= 2` |
| Decision-time predictors | Product category and price, delivery lateness, carrier, service level, customer history, review language |
| Validation split | Temporal split on the order timestamp |
| Evaluation metric | Precision and recall on the low-rating class, reported with the base rate, because the class is the minority and accuracy would be uninformative |
| Leakage, fairness or deployment risk | **Selection bias:** only reviewed items appear at all — 7,000 reviews against 15,685 items — so the model learns the behaviour of customers who chose to review. `review_body_clean`, `review_length_chars` and `helpful_votes` are all recorded with the rating and are not available before it. |

### MLQ-4 (Yandu): What natural customer segments exist in purchasing behaviour?

| Element | Response |
|---|---|
| EDA evidence | *Cite the figure or finding and its quantified observation* |
| Business decision | *Replace* |
| Problem type and analysis unit | clustering · one customer |
| Target or unsupervised objective | No target. Objective: compact, well-separated groups on spend, frequency, recency, category mix and channel |
| Decision-time predictors | Order count, total and average order value, category shares, channel shares, signup recency, marketing consent |
| Validation split | No train/test split; stability is assessed by re-clustering bootstrap resamples and comparing assignments |
| Evaluation metric | Silhouette score for separation, plus a stability score across resamples |
| Leakage, fairness or deployment risk | With 500 customers the clusters may not be stable; features on different scales will dominate the distance unless standardised. **Fairness:** if a segment is used to price or to allocate service, check it is not a proxy for postcode. |

### MLQ-5 (Echo): What order volume should be planned for next month?

| Element | Response |
|---|---|
| EDA evidence | *Cite the figure or finding and its quantified observation* |
| Business decision | *Replace* |
| Problem type and analysis unit | forecasting · one month, at the national level |
| Target or unsupervised objective | Monthly order count (and, as a second target, monthly revenue) |
| Decision-time predictors | Historic monthly volume, month of year, channel mix, promotion intensity |
| Validation split | Rolling-origin evaluation — fit to months 1..n, predict n+1, roll forward. A random split is invalid for a time series |
| Evaluation metric | Mean absolute percentage error against a seasonal-naive baseline, which is the only honest benchmark for a first forecast |
| Leakage, fairness or deployment risk | **Only one year of data**, so seasonality cannot be separated from trend and the model has no second cycle to learn from. Deliveries running into January 2019 belong to December orders and must not be counted as January demand. |

## 5. Limitations and conclusion

Summarise the most decision-relevant limitations, what the data cannot establish
and the next evidence needed. End with a concise conclusion rather than new
analysis.


*Owner: Jasmine. The most decision-relevant limitations, what this data cannot establish, and
the next evidence needed — then a short conclusion rather than new analysis.*

Starting points that are already evidenced rather than speculative:

- **One year, one region.** Seasonality cannot be separated from trend.
- **Three columns have no variance** — `order_status`, `delivery_status` and
  `verified_purchase` are single-valued across every row, so no analysis can use them.
- **Reviews are self-selected.** 7,000 reviews against 15,685 order items; every
  review-based finding describes reviewers, not customers.
- **Association is not causation** anywhere in this report — in particular between lateness
  and rating, where an unobserved product or seller effect could drive both.

## References

Use one consistent referencing style for external facts, code, data and ideas.
